# Exercise 1 · Exploring EO missions and metadata
**ICAT3370 Foundations of Space Data and EO Systems** · 30 September 2026 · 2 h

SYKE, the Finnish Environment Institute, runs **Tarkka** ([map viewer](https://tarkka.syke.fi/eo-tarkka/map/?ver=0&lang=en&time=2026-06-21&style=opt&bbox=21.07674%2C62.94579%2C21.98357%2C63.37267&coll=call&data=d-bm-esri%2Cd-s2)), a public service that turns satellite images into water quality maps of Finnish lakes and the Baltic Sea. Authorities use these maps for water status reporting.

Your task throughout this course will be:

> *Could we make a Tarkka like turbidity map of the Vaasa and Kvarken coast ourselves, from free satellite data and what must we know about the data before we trust it?*

We haven't chosen an image and before that a satellite. Today we are trying to tackle this problem.

**What you will learn today**

- What spatial, spectral and temporal resolution mean, and how the pixel size of a satellite image is decided by the instrument and the orbit.
- Which Copernicus Sentinel missions exist, what each of them measures, and how to compare missions against the requirements of a task.
- The terms on a mission card, such as swath, sun-synchronous orbit, repeat cycle and revisit time, and what they mean.
- How to find a Sentinel-2 image in the Copernicus Browser, and how to read what its product name says.
- What a Sentinel-2 product contains, which files describe it, and how to read the acquisition metadata from them.
- That every pixel carries one reflectance value per band, and how two images of the same place can be compared.
- How to start an EO Data Diary that records what was measured and what was chosen for one dataset.

**Instructions.** Run a cell with **Shift + Enter**. Feel free to change something where the line is marked `# ✏️ EDIT`. Same goes for wirting an answer where the cell marked **✍️ Your answer**, type, and press Shift + Enter to close it.

---
## 0 · Tarkka and the course question

This is the coast of Vaasa on **21 June 2026**, as Tarkka shows it. The city is the pale patch on the right. The bay in front of it, Eteläinen Kaupunginselkä, is brown. The Laihianjoki and Sulvanjoki rivers carry mud into it from the south, and that mud is what Tarkka measures as turbidity. The round patchwork of fields to the south is Söderfjärden, a meteorite crater.

<img src="figures/tarkka_20260621_opt.jpg" style="max-width:100%;border:1px solid #ccc">

*Tarkka map viewer, Sentinel-2, 21.06.2026, style "Optimized". [Open this view yourself](https://tarkka.syke.fi/eo-tarkka/map/?ver=0&lang=en&time=2026-06-21&style=opt&bbox=21.07674%2C62.94579%2C21.98357%2C63.37267&coll=call&data=d-s2%2Cd-bm-esri). Contains modified Copernicus Sentinel-2 data, Syke (2026).*


Here is the same place three months later, and then the same September image shown a second time.

<img src="figures/tarkka_three_views.jpg" style="max-width:100%;border:1px solid #ccc">

The left picture is 21 June. The middle one is 27 September, from the same website with the same settings. The right one is the same 27 September image, from the same satellite pass and the same pixels. The only thing that changed between the middle and the right picture is one dropdown in Tarkka, from "Optimized" to "Natural".

Three questions. We will not answer them today :D
1. What happened to the coast between June and September?
2. What happened between the middle and the right picture, when nothing on the ground changed at all?
3. Which of the three pictures shows what the satellite measured?

By the end of the course you will be able to answer all of them.

**Task 0.** Look at the image. Find **one thing** here that you would want to monitor from space, and say for whom it matters.

✍️ **Your answer** (one sentence):



---
## 1 · Requirements, what the satellite must measure

Before we can pick a satellite, we need to know what we are asking of it.

Four questions:

**1. What is the map actually showing?**
The map shows brown water next to blue water, which is a difference in *colour*. What the satellite actually records is the sunlight reflected from the water, in a few bands. Colour is how we show three of those bands on a screen. Where does that light come from, and what does that say about the time of day and the weather?

**2. How small are the things you can see?**
Use the scale bar. How wide is the river plume? How wide is a channel between two islands in the archipelago? If a satellite pixel were as wide as the whole plume, what would the map look like?

**3. How often would you need a new map?**
A plume changes after heavy rain. An algae bloom can appear within days. If you got one map per year, would it be useful? One per month? One per week?

**4. Who pays for this, and for how long?**
Tarkka covers all of Finland and has run for years. What does that mean for the price of the images and for the satellite itself?


**Task 1.** Fill in the table. Use your own words.

✍️ **Your answer**

| The map needs a satellite that can … | In numbers, or in a word | Why (from the map) |
|---|---|---|
| 1. see … | | |
| 2. see things as small as … | | |
| 3. come back every … | | |
| 4. cost … and last … | | |


### Spatial, spectral and temporal resolution

**Spectral resolution.** Sunlight is not one thing but a whole range of wavelengths, and you have been introduced it as the electromagnetic spectrum. What you may not have met yet is what it has to do with a satellite. A satellite instrument does not record the whole range. It records a few chosen slices of it and each slice is called a **band**. The colours your eye sees are three narrow slices at the short wavelength end. Everything else on the axis is light that your eye cannot see but an instrument can, and the slices a mission chooses decide what it can and cannot tell apart.

The tool below is NASA's band comparison tool. Each coloured bar is one band of one instrument, drawn on the wavelength axis, over a grey curve that shows how much light the atmosphere lets through at each wavelength. Notice that Sentinel-2 and Landsat have almost the same slices, that Sentinel-3 has many more narrow ones, and that **almost** all of them sit inside the "windows" where the grey curve is high.


In [ ]:
from IPython.display import IFrame
IFrame("https://science.nasa.gov/specials/band-comparison-tool/?layers=sentinel2,sentinel3-olci,landsat8-9,modis",
       width="100%", height=850)

If the tool does not load, this is the same idea, the spectrum from 0.4 µm to 1 mm, with the atmospheric windows and the three visible slices marked B, G, R at the left end.

<img src="figures/books/campbell_fig2-3_em_spectrum.png" style="max-width:85%">
<br><small>Campbell &amp; Wynne, <i>Introduction to Remote Sensing</i>, 6th ed., Fig. 2.3.</small>


**Spatial resolution.** How small a thing the satellite can resolve, given by the size of one pixel on the ground. Below is the same view of Vaasa from three real satellites. Sentinel-2 with 10 m pixels, Landsat 8/9 with 30 m pixels and Sentinel-3 OLCI with 300 m pixels. Drag the handle across the picture. The left side stays Sentinel-2 and the right side shows the satellite you choose from the menu. Watch what happens to the city, the harbour and the bay. Then decide for yourself at which pixel size the plume you looked at in first part would still be visible.


In [ ]:
from helpers import swipe

swipe({"Sentinel-2, 10 m pixels": "figures/pixel_zoom_s2.jpg",
              "Landsat 8/9, 30 m pixels": "figures/pixel_zoom_landsat.jpg",
              "Sentinel-3 OLCI, 300 m pixels": "figures/pixel_zoom_s3.jpg"})

In [ ]:
# The same three satellites, zoomed out to the whole coast. At this scale the 300 m pixels start to make sense.
swipe({"Sentinel-2, 10 m pixels": "figures/pixel_wide_s2.jpg",
              "Landsat 8/9, 30 m pixels": "figures/pixel_wide_landsat.jpg",
              "Sentinel-3 OLCI, 300 m pixels": "figures/pixel_wide_s3.jpg"})

*All three from the Tarkka map viewer, 21 June 2026 for Sentinel-2 and the nearest clear days for the other two. Contains modified Copernicus Sentinel-2 and Sentinel-3 data, USGS/NASA Landsat, Syke (2026).*


### What sets the pixel size

#### 1 · The geometry

An imaging instrument works like a camera. Light from a patch of ground passes through the telescope and lands on one **detector element**, one light sensitive cell of the sensor (also called a sensor pixel). The patch on the ground and the detector element form two similar triangles with their apex at the telescope, so their sizes are in the same ratio as their distances from it:

$$\frac{G}{H}=\frac{p}{f}\qquad\Rightarrow\qquad G=\frac{p\,H}{f}$$

where $G$ is the size of one pixel on the ground, $H$ the altitude of the satellite, $p$ the size (pitch) of one detector element and $f$ the focal length of the telescope. The angle $p/f$ is the **instantaneous field of view**, IFOV. In Lillesand's notation the same relation reads $D = H'\beta$ (Eq. 4.1).

<img src="figures/books/lillesand_fig4-2_ifov.png" style="max-width:55%">
<br><small>Lillesand, Kiefer &amp; Chipman, <i>Remote Sensing and Image Interpretation</i>, 7th ed., Fig. 4.2. One detector element looks down through the angle $\beta$ (the IFOV) from height $H'$ and sees a ground resolution element of size $D$. In our notation $D = G$ and $H' = H$.</small>

**Sentinel-2 MSI.** Detector pitch 7.5 µm for the 10 m bands, altitude 786 km, focal length about 0.6 m.

$$G=\frac{7.5\times10^{-6}\ \mathrm{m}\times 786\,000\ \mathrm{m}}{0.6\ \mathrm{m}}=9.8\ \mathrm{m}$$

The ground is about 1.3 million times further from the telescope than the detector is, so each detector element is projected onto the ground 1.3 million times larger. The 20 m bands use detector elements of 15 µm on the same telescope in the same orbit, and the 60 m bands use those same 15 µm elements and add up 3 by 3 blocks of them on board, because the atmospheric quantities they measure vary slowly and do not need fine pixels.

#### 2 · The same geometry gives the swath

The whole line of detectors sees an angle called the field of view, FOV, 20.6° for MSI.

$$S = 2H\tan\left(\frac{\mathrm{FOV}}{2}\right)=2\times786\ \mathrm{km}\times\tan(10.3^\circ)=286\ \mathrm{km}$$

which is the 290 km in the mission table. This is step 1 applied to all 31 152 detector elements at once. The 10 m bands have 31 152 elements in one row across the swath, arranged in twelve modules (ESA calls the modules "detectors") whose ends overlap slightly, so 31 152 × 9.8 m is about 305 km and the usable swath 290 km. Halving the pixel halves the swath unless the number of detectors is doubled, and doubling the detectors doubles the data.

#### 3 · Why the detector elements cannot be made as small as you like

The formula in step 1 says that smaller detector elements give smaller pixels. That is true only up to a point, because a telescope, like any lens, can only make an image so sharp. Zoom into a phone photo and beyond a certain point you see no new detail, only bigger blur, and adding more pixels to the screen would not help. Sentinel-2's telescope gives a picture that is sharp down to about 4 m on the ground in red band. Pixels smaller than that would only divide the blur into smaller pieces, so the designers chose 10 m, well above the limit, and kept the strip wide and the pixels bright instead. In the infrared bands the blur is about 14 m, which is why those bands are sampled at 20 m.

How sharp a telescope can be depends on the size of its mirror and on the wavelength of the light. The smallest angle it can separate is

$$\theta = 1.22\,\frac{\lambda}{D}$$

with $\lambda$ the wavelength and $D$ the mirror diameter. On the ground that angle becomes $\theta \times H$. For Sentinel-2, $D$ = 150 mm and $\lambda$ = 0.665 µm give 5.4 µrad, which is 4.3 m from 786 km. The satellites that reach 30 cm carry a mirror over a metre wide, a much longer focal length and fly lower.

#### 4 · How the picture is built up, the pushbroom

The sensor in Sentinel-2 is a single row of detector elements. The satellite flies forward and the row sweeps over the ground like a broom pushed across a floor, recording one line of pixels after another. This is a **pushbroom** instrument. Across the track the pixel size comes from the geometry above. Along the track it is how far the satellite moves during one exposure. The older design, a **whiskbroom**, has one detector and a mirror that sweeps it from side to side (Landsat 5, MODIS).

<img src="figures/books/campbell_fig5-1_pushbroom_vs_whiskbroom.png" style="max-width:70%">
<br><small>Campbell &amp; Wynne, <i>Introduction to Remote Sensing</i>, 6th ed., Fig. 5.1. Left (a) is the pushbroom, a linear array sliding forward. Right (b) is the whiskbroom, a scan mirror sweeping one detector from side to side.</small>

**Pixel size is the sampling grid, not the smallest thing you can see.** A 2 m road shows in a 10 m image because it is much brighter than the field around it. Two boats 10 m apart do not separate. A working rule is that the pixel should be less than half the size of the feature you want to identify (Jensen, ch. 1).

Sources for the Sentinel-2 numbers are eoPortal, "Copernicus Sentinel-2" (detector pitch, pupil diameter, field of view), and SentiWiki S2 Mission (orbit, swath, band resolutions).


**Task 1b.** The sliders start at Sentinel-2's values. Make the pixel 1 m in two different ways, first by moving the satellite lower and lengthening the focal length, then by shrinking the detector elements. Watch what happens to the width of the strip each time.

In [ ]:
from helpers import pixel_size_calculator
pixel_size_calculator()

**Temporal resolution.** How often the satellite comes back to the same place. A single image cannot show change. Only a series can. Below is a site one kilometre north of the market square, Wärtsilä's old engine works on the Onkilahti shore in Vöyrinkaupunki, which the company left when it moved to Vaskiluoto. The city is turning the ten hectares into a new residential district, and during 2025 the largest hall on the site, about 7,400 square metres of floor, was demolished.

The yellow box is the Wärtsilä site and the blue box is Strömberg Park, which we come to in a moment.

<img src="figures/vaasa_locator_sites.jpg" style="max-width:70%">
<br><small>Sentinel-2, 21 June 2026. Contains modified Copernicus Sentinel-2 data (2026).</small>

Now the site itself, 1.2 km across, in five Sentinel-2 images over three years. The left side is always July 2023. Choose the later image from the menu and drag the handle across.


In [ ]:
swipe({"15 July 2023": "figures/wartsila_20230715.png",
              "14 July 2025": "figures/wartsila_20250714.png",
              "13 August 2025": "figures/wartsila_20250813.png",
              "21 June 2026": "figures/wartsila_20260621.png",
              "2 September 2026": "figures/wartsila_20260902.png"}, width=700)

The hall is intact in 2023. In July 2025 its roof is half open and bright rubble shows in the middle. By August most of it is gone, and in June 2026 the whole footprint is bare sand. The last image is from four weeks ago.

A second site shows what the return visits are worth when something happens suddenly. On 29 March 2023 a fire destroyed part of Hitachi Energy's transformer factory at Strömberg Park. Sentinel-2 had photographed the site three days before, under snow. It passed over again on the day of the fire but the site was under cloud, so that image shows nothing. Two days later the next pass was clear and the burnt section is  visible as a black patch among the snow covered roofs.

<img src="figures/hitachi_fire_march2023.jpg" style="max-width:100%">
<br><small>Sentinel-2 L2A, 26, 29 and 31 March 2023, 1.4 km across. Contains modified Copernicus Sentinel-2 data (2023).</small>

How often a satellite comes back, and why it is five days for Sentinel-2 and not one or thirty, is the subject of Exercise 2.

**Cost and delivery.** No picture is needed for this one. It is whether the data is free or paid for, how quickly it reaches you, and whether the satellite will still be flying next year.


### The numbers behind the picture

Everything above was about pictures. Underneath every picture there are numbers, one per band for every pixel, and those numbers are what the whole course computes with. The cell below is our 21 June image, 6 km around Vaasa. Click anywhere to place a point. The table lists the reflectance of that pixel in each of the twelve bands, and the plot draws them against wavelength, which is the **spectral signature** of that spot. Place a few points, in the brown bay, in clear water further out, on the forest, on the city, and compare the curves. Below the plot you can switch on the spectra of known surfaces, water, grass, soil, concrete and snow, drawn dashed, to see which of them your points resemble. These are the same reference curves the Copernicus Browser shows in its spectral explorer.

Do not try to explain the curves today. In Exercise 3 you will read them, and by then you will know what each band sees.


In [ ]:
from helpers import spectral_probe
spectral_probe()

### Change between two dates, the Strömberg Park fire

Two Sentinel-2 images of the same 2 km around Strömberg Park, 26 March 2023, three days before the fire, and 31 March, two days after. Click on either image to place a point; it appears on both, and the plot draws the spectrum of that pixel on both dates, solid for before and dashed for after. Place one point on the burnt part of the hall, one on an untouched roof next to it, and one on a snow-covered field, and compare how each pair of curves changed.

The menu above the images offers three views. The last choice shows the plain difference of the red band between the two dates, after minus before, so that what changed stands out in red or blue and what did not is green.

Some of the change between the two dates is snow melting, not fire. Which of your points show that, and how can you tell it apart from the burnt roof?


In [ ]:
from helpers import change_probe
change_probe()

Resolution is a vector, not a ranking. No mission is best at all of these.

---
## 2 · Copernicus Sentinel missions and mission selection

Before we compare satellites, meet the ones Europe has. **Copernicus** is the European Union's Earth observation programme. Its satellites are called **Sentinels** and there are six families of them, each built to measure a different thing.

In [ ]:
from IPython.display import Video
Video("https://dlmultimedia.esa.int/download/public/videos/2017/06/018/1706_018_AR_EN.mp4", width=760)

### The Sentinels in the Copernicus Browser

The Copernicus Browser is the public viewer for all Sentinel data, [browser.dataspace.copernicus.eu](https://browser.dataspace.copernicus.eu/). It asks you to log in, and "Use anonymously" is enough for everything in this exercise. In the panel on the left you can choose the mission and the date and the map shows what that Sentinel records.


### The Copernicus missions in one overview

**A, B, C, D.** The letters are not different satellites types but units of the same mission, launched one after another. Sentinel-2A launched in 2015, 2B in 2017, 2C in 2024, and 2D is expected to launch in 2028. Two of them fly at a time on the same orbit, so that between them they pass over every place twice as often as one satellite could. The figure below shows the three Sentinel-2 satellites on their shared orbit and where each one is on each day of the 10 day cycle.

<img src="figures/esa_s2_constellation_cycle.jpg" style="max-width:60%">
<br><small>Sentinel-2A, 2B and 2C on the same orbit, and their positions through one 10 day cycle. Credit ESA, SentiWiki.</small>

<img src="figures/copernicus_timeline.png" style="max-width:100%">
<br><small>Copernicus Sentinel missions, 2015 to 2030. Credit ESA / EU Copernicus. Mission details for each one at [eoportal.org/satellite-missions/copernicus](https://www.eoportal.org/satellite-missions/copernicus).</small>


**Task 2** Five Sentinels, five things they measure. Match them. Use what you just saw and, if you need to, the official mission pages at [sentinels.copernicus.eu/missions](https://sentinels.copernicus.eu/missions), where each mission has a card saying what it observes. Choose for every row, press **Check**, and fix the red ones. A correct row plays ESA's short animation of that satellite.


In [ ]:
from helpers import sentinel_match
sentinel_match()

**Live position.** Sentinel-2A is somewhere over the Earth at this moment. [This page](https://satellitemap.space/sat/40697) shows its position and the path it is flying. Have a look for thirty seconds. Why the path looks the way it does is Exercise 2.

Below, each row is one mission. The columns are your four requirements from part 1, what it measures, how small a thing it can see, how often it comes back, and what it costs. One extra column, **strip width** (swath), tells how wide a band of ground the satellite sees in one pass.


In [ ]:
from helpers import load_missions, plot_trade_off

missions = load_missions()
missions

**Task 2a.** Go through your four requirements. For each requirement, cross out every mission that fails it and write down *which requirement* eliminated it.

✍️ **Your answer** (edit the table)

| Mission | Still in? | Eliminated by which requirement (1 to 4)? |
|---|---|---|
| Sentinel-2 | | |
| Sentinel-3 OLCI | | |
| Sentinel-1 | | |
| Sentinel-5P | | |
| Landsat 8/9 | | |
| MODIS | | |
| PlanetScope | | |
| WorldView-3 | | |
| ICEYE | | |

How many survive?


In [ ]:
# Task 2b. The same elimination, done by the computer.
# Every filter starts wide open. Set each one to what YOU wrote in Task 1, then run the cell.

MUST_MEASURE   = "any"     # ✏️ EDIT: "reflected", "radar" or "gases"
MAX_PIXEL_M    = 100000    # ✏️ EDIT: largest pixel that still shows the plume, in metres
MAX_DAYS       = 100000    # ✏️ EDIT: longest gap between visits you can accept, in days
MUST_COST      = "any"     # ✏️ EDIT: "free" or "paid"

shortlist = missions
if MUST_MEASURE != "any":
    shortlist = shortlist[shortlist["what it measures"].str.startswith(MUST_MEASURE)]
shortlist = shortlist[shortlist["pixel size (m)"] <= MAX_PIXEL_M]
shortlist = shortlist[shortlist["days between visits"] <= MAX_DAYS]
if MUST_COST != "any":
    shortlist = shortlist[shortlist["cost"] == MUST_COST]

print(len(shortlist), "of", len(missions), "missions pass your four requirements")
shortlist

In [ ]:
plot_trade_off(missions)

**Questions**

1. Which missions survived, and which single requirement removed most of the others?
2. Tarkka uses Sentinel-2 **and** Sentinel-3. You probably crossed Sentinel-3 out. Look at where it sits on the plot. What does it give that Sentinel-2 cannot, and where on the Vaasa coast would that matter?
3. Why is the bottom-left corner of the plot empty (very small pixels **and** daily visits **and** free)?
4. Sentinel-1 sees through clouds, and our summers have clouds. You crossed it out on requirement 1 because it does not record reflected sunlight. Was that too quick? *(No answer needed today. We come back to it in Exercise 4.)*

✍️ **Your answers:**



Sentinel-2 won on what it measures, pixel size, revisit and cost. Here are its thirteen bands, with the pixel size of each. Four of them are 10 m, blue, green, red and one near-infrared. In Exercise 3 you will use these bands yourself.

<img src="figures/books/lillesand_table5-15_sentinel2_bands.png" style="max-width:80%">
<br><small>Lillesand, Kiefer &amp; Chipman, <i>Remote Sensing and Image Interpretation</i>, 7th ed., Table 5.15.</small>

Write one sentence in your own words. *We use Sentinel-2 because … , and not … because …* You will put it in your Diary at the end.

✍️ **Your sentence:**



### The Sentinel-2 mission card

Every mission has a page like this, on SentiWiki or eoPortal, and the same words appear on all of them. Here they are for Sentinel-2, one sentence each. Today we only read the numbers. Where they come from, and what they mean for Vaasa, is Exercise 2.

| Term | Sentinel-2 | Meaning |
|---|---|---|
| Mean altitude | 786 km | Height of the orbit above the Earth's surface. |
| Orbital period | 101 minutes | Time for one trip around the Earth, about 14 a day. |
| Inclination | 98.6° | Angle between the orbit and the equator. Close to 90° means the satellite passes near both poles and sees every latitude. |
| Sun-synchronous orbit | yes | The orbit plane turns with the Sun through the year, so the satellite crosses any place at the same local solar time every day. This is why every Vaasa image is taken near 10:10 UTC and why the light is comparable from one image to the next. |
| Local time at the equator | 10:30 | The local solar time when the satellite crosses the equator flying south, chosen as the compromise between morning cloud and good light. |
| Repeat cycle | 10 days (5 with two satellites) | After this many days the satellite flies exactly the same track again. |
| Revisit time | 2 to 3 days at Vaasa | How often a place is imaged. Shorter than the repeat cycle where neighbouring strips overlap, which they do more and more toward the poles. |
| Swath | 290 km | Width of the strip imaged in one pass. |
| Satellites in orbit | 2A, 2B, 2C | Copies of the same design, two active at a time, half an orbit apart. |
| Instrument | MSI, pushbroom, 13 bands | What the satellite carries and how it builds the image. |

<table style="width:100%"><tr>
<td style="width:48%;vertical-align:top"><img src="figures/esa_s2_sun_synchronous_orbit.png" style="max-width:100%"><br><small>The sun-synchronous orbit seen from above the North Pole and from the equator. The orbit plane keeps the same angle to the Sun all year, and the satellite crosses the equator southbound at 10:30 local time. Credit ESA, SentiWiki.</small></td>
<td style="width:52%;vertical-align:top"><img src="figures/esa_s2_revisit_map.jpg" style="max-width:100%"><br><small>Where the two-satellite constellation returns within 5 days (green) and 10 days (yellow). All land and coastal waters are covered every 5 days at the equator, and more often further north. Credit ESA, SentiWiki.</small></td>
</tr></table>

Sources are SentiWiki, S2 Mission, and eoPortal, Copernicus Sentinel-2. ESA's short film below shows how the two satellites sweep the whole Earth in strips.


In [ ]:
from IPython.display import Video
Video("https://dlmultimedia.esa.int/download/public/videos/2016/08/004/1608_004_AR_EN.mp4", width=760)
# ESA, "Sentinel-2 global coverage" (2016). Credit ESA/ATG medialab.

---
## 3 · Find the image

You have chosen the satellite. Now find the one image of it that we will use for the rest of the course. You do this yourself, in the Copernicus Browser, with the requirements you wrote in part 1 as your search filters.

**Task 3.** Open [browser.dataspace.copernicus.eu](https://browser.dataspace.copernicus.eu/) and choose "Use anonymously".

1. Type Vaasa in the place search at the top right and let the map move there.
2. Open the **Search** tab in the panel on the left.
3. Tick **Sentinel-2**. Under it, choose **L2A**.
4. Set the time range to **1 June 2026 to 30 June 2026**.
5. Set the **cloud cover** limit to a value of your own choosing.
6. Press **Search** and look through the results. Click one, then **Visualize**, and look at the bay in front of the city. Is the brown plume visible?
7. Find a day on which it is. Then copy the **product name** of that result, the long string that starts with `S2`, into the box below. Copy only the name.

✍️ **The product I found**

| | |
|---|---|
| Date of the image | |
| Cloud cover shown in the Browser | |
| Product name | |

**Task 3b.** For the rest of the course everyone must work on the same image. If your day was different, search once more with the time range set to that single day and copy that product name too.

You will see two products for 21 June with the same time, one whose name contains `L1C` and one with `L2A`. One of them still has the atmosphere in the numbers and the other has it removed. Which one we want, and why, is Exercise 5.

✍️ **21 June 2026, both products**

| Level | Product name |
|---|---|
| L1C | |
| L2A | |


---
## 4 · What the product says about itself

### 4.1 · The product is a folder

What you found in the Browser is called a **product**. It is not one image file. ESA delivers it as a folder in a fixed layout called **SAFE** (Standard Archive Format for Europe), with one file per band and several files that describe the data. The cell below prints that layout for our 21 June product.


In [ ]:
from helpers import show_safe_structure, parse_product_name, print_metadata_lines, read_metadata

SCENE_L1C = "S2B_MSIL1C_20260621T101019_N0512_R022_T34VER_20260621T134515"
SCENE_L2A = "S2B_MSIL2A_20260621T101019_N0512_R022_T34VER_20260621T140048"

show_safe_structure(SCENE_L2A)

Three things to notice. The thirteen bands are thirteen separate files, sorted by pixel size. A "true colour" picture, such as the one Tarkka shows, is made from three of them, B04, B03 and B02. And two of the files are not images at all but descriptions of the images, `MTD_MSIL2A.xml` for the whole product and `MTD_TL.xml` for the tile. Those two are where we go next, after the name.


### 4.2 · Reading the product name

The naming convention of Sentinel-2. The rule is published by ESA on SentiWiki, [Sentinel-2 Products](https://sentiwiki.copernicus.eu/__attachments/a_0d279c523381917992e4b2b28f5229995e592ffc6e17fc1aebf09b8c120240ea/S2-PDGS-CS-DI-PSD%20-%20S2%20Product%20Specification%20Document%202024%20-%2015.0.pdf), which also describes everything else inside a product. Here it is.

| Field | Format | Meaning |
|---|---|---|
| 1 | `S2A`, `S2B`, `S2C` | Mission and which satellite of the series |
| 2 | `MSIL1C` or `MSIL2A` | Instrument (MSI) and processing level |
| 3 | `YYYYMMDDTHHMMSS` | Sensing start time, in UTC, when the picture was taken |
| 4 | `Nxxyy` | Processing baseline, the version of the software that produced the file |
| 5 | `Rxxx` | Relative orbit number, which of the 143 ground tracks the satellite was on |
| 6 | `Txxxxx` | Tile, a 100 km square of the military grid (UTM zone, latitude band, square) |
| 7 | `YYYYMMDDTHHMMSS` | Product generation time, in UTC, when the file was made |

**Task 4a.** Decode the L2A name you copied. Write what each field says, in words.

`S2B` · `MSIL2A` · `20260621T101019` · `N0512` · `R022` · `T34VER` · `20260621T140048`

✍️ **Your reading**

| Field | What it says |
|---|---|
| S2B | |
| MSIL2A | |
| 20260621T101019 | |
| N0512 | |
| R022 | |
| T34VER | |
| 20260621T140048 | |

Two questions. The name has two timestamps. How far apart are they, and what happened in between? And where in the name is Vaasa?


In [ ]:
parse_product_name(SCENE_L2A)

Now the L1C name of the same day, `S2B_MSIL1C_20260621T101019_N0512_R022_T34VER_20260621T134515`. Compare the two names field by field. Which fields are the same, and which differ? The sensing time is identical because it is the same pass of the same satellite. The level differs, and so does the generation time, because the L2A product was made from the L1C product some minutes later.


### 4.3 · Reading the metadata files

The two XML files describe the images. Together they are 6 000 lines long, so the cell below prints only the lines that carry five tags, with their line numbers in the file. 

**Task 4b.** From the printed lines, find and write down four values.


In [ ]:
print_metadata_lines(SCENE_L2A)

✍️ **From the files**

| Value | From which file | What you found |
|---|---|---|
| Sensing time | | |
| Processing baseline | | |
| Cloud cover, % | | |
| Mean sun zenith angle, ° | | |

Three checks. Does the baseline in the file match the `N0512` in the name? Does the cloud cover in the file match what the Browser showed you in part 3? And the mean sun zenith angle is an **acquisition parameter** we have not talked about.

In [ ]:
# The same values
read_metadata(SCENE_L2A)

### 4.4 · The EO Data Diary

**Task 4c, first part.** Before you look further down. In November you will hand this image to a colleague who has to trust the map you made from it. Write five things they would need to know about the image.

✍️ **Five things**

1.
2.
3.
4.
5.

**Task 4c, second part.** Now the Diary. It is one page per dataset, and you will keep it through the whole course. Compare your five things with its rows, then fill in every row. Later exercises add rows of their own to the same page.

✍️ **EO Data Diary, Sentinel-2 image of 21 June 2026**

| Section | Field | Your entry |
|---|---|---|
| Identity | Mission and satellite | |
| | Why this mission | |
| | Product name | |
| | Where found | |
| | Processing level | |
| | Processing baseline | |
| Acquisition | Sensing time, UTC | |
| | Cloud cover, % | |
| | Mean sun zenith angle | |
| Sampling | Pixel size of the bands | |
| | Swath | |
| Orbit | Relative orbit, tile | |

---
## Wrap-up

Write one sentence into the Diary row "Why this mission", in your own words. *We use Sentinel-2 because … , and not … because …*

**What comes next.** The product name contained `R022` and the time 10:10. Why every image of Vaasa is taken at that time of day, why the satellite flies that track, and how many days pass until the next image, is Exercise 2 on 6 October.
